In [0]:
%pip install nltk

In [0]:
%restart_python

In [0]:
import nltk
import ipywidgets as widgets
from IPython.display import display, clear_output
nltk.download('wordnet')
from nltk.corpus import wordnet
import random

In [0]:
class WordGuessingGame:
    def __init__(self):
        self.words = []
        self.current_word_idx = 0
        self.current_word = ""
        self.guessed_word = ""
        self.letters_left = []
        self.tries = 0
        self.game_started = False
        self.letter_buttons = {}
        
        # Setup UI
        self.setup_ui()
        
    def setup_ui(self):
        # Game setup widgets
        self.word_length_input = widgets.IntText(value=5, description='Word Length:', min=3, max=15)
        self.num_words_input = widgets.IntText(value=3, description='Num Words:', min=1, max=10)
        self.start_btn = widgets.Button(description='Start Game', button_style='success')
        self.start_btn.on_click(self.start_game)
        
        # Game play widgets
        self.game_info = widgets.HTML(value="<h3>Configure game settings and click Start Game</h3>")
        self.word_display = widgets.HTML(value="")
        
        # Create letter buttons (A-Z keyboard)
        self.create_letter_buttons()
        
        # Alternative text input (optional)
        self.letter_input = widgets.Text(description='Type:', placeholder='Or type a letter', disabled=True)
        self.guess_btn = widgets.Button(description='Guess', button_style='primary', disabled=True)
        self.guess_btn.on_click(lambda x: self.make_guess(self.letter_input.value))
        self.letter_input.on_submit(lambda x: self.make_guess(self.letter_input.value))
        
        self.message_display = widgets.HTML(value="")
        self.stats_display = widgets.HTML(value="")
        
        # Layout
        setup_box = widgets.VBox([self.word_length_input, self.num_words_input, self.start_btn])
        game_box = widgets.VBox([
            self.game_info,
            self.word_display,
            self.keyboard_box,
            widgets.HTML("<p style='text-align:center; margin:10px 0;'><i>Or type below:</i></p>"),
            widgets.HBox([self.letter_input, self.guess_btn]),
            self.message_display,
            self.stats_display
        ])
        
        self.main_ui = widgets.VBox([setup_box, widgets.HTML("<hr>"), game_box])
        
    def create_letter_buttons(self):
        """Create clickable letter buttons in QWERTY layout"""
        # QWERTY keyboard layout
        rows = [
            'QWERTYUIOP',
            'ASDFGHJKL',
            'ZXCVBNM'
        ]
        
        keyboard_rows = []
        for row in rows:
            row_buttons = []
            for letter in row:
                btn = widgets.Button(
                    description=letter,
                    layout=widgets.Layout(width='40px', height='40px'),
                    button_style='',
                    disabled=True
                )
                btn.on_click(lambda b, l=letter.lower(): self.make_guess(l))
                self.letter_buttons[letter.lower()] = btn
                row_buttons.append(btn)
            keyboard_rows.append(widgets.HBox(row_buttons, layout=widgets.Layout(justify_content='center')))
        
        self.keyboard_box = widgets.VBox(keyboard_rows, layout=widgets.Layout(
            border='1px solid #ccc',
            padding='10px',
            border_radius='10px',
            margin='10px 0'
        ))
        
    def start_game(self, btn):
        word_len = self.word_length_input.value
        tot_words = self.num_words_input.value
        
        # Generate words
        all_words = [x for x in list(set([w.name().split('.')[0] for w in wordnet.all_synsets()])) 
                     if ('_' not in x) and (len(x) == word_len)]
        
        if len(all_words) < tot_words:
            self.game_info.value = f"<h3 style='color:red;'>Not enough {word_len}-letter words! Found {len(all_words)}.</h3>"
            return
            
        self.words = random.sample(all_words, tot_words)
        self.current_word_idx = 0
        self.game_started = True
        
        # Disable setup controls
        self.word_length_input.disabled = True
        self.num_words_input.disabled = True
        self.start_btn.disabled = True
        
        # Enable game controls
        self.letter_input.disabled = False
        self.guess_btn.disabled = False
        for btn in self.letter_buttons.values():
            btn.disabled = False
        
        self.start_new_word()
        
    def start_new_word(self):
        if self.current_word_idx >= len(self.words):
            self.end_game()
            return
            
        self.current_word = self.words[self.current_word_idx]
        self.guessed_word = '_' * len(self.current_word)
        self.letters_left = list('abcdefghijklmnopqrstuvwxyz')
        self.tries = 0
        
        # Reset all letter buttons
        for letter, btn in self.letter_buttons.items():
            btn.disabled = False
            btn.button_style = ''
        
        self.update_display()
        self.game_info.value = f"<h3>Word {self.current_word_idx + 1} of {len(self.words)}</h3>"
        self.message_display.value = "<p style='color:blue;'>New word! Click a letter or type to guess...</p>"
        
    def make_guess(self, letter):
        if isinstance(letter, str):
            letter = letter.lower().strip()
        else:
            return
        
        if not letter or len(letter) != 1 or not letter.isalpha():
            self.message_display.value = "<p style='color:red;'>Please enter a single letter!</p>"
            return
            
        if letter not in self.letters_left:
            self.message_display.value = "<p style='color:orange;'>You already guessed that letter!</p>"
            return
            
        self.letters_left.remove(letter)
        self.tries += 1
        
        # Update button state
        if letter in self.letter_buttons:
            self.letter_buttons[letter].disabled = True
        
        if letter in self.current_word:
            # Visual feedback on button
            if letter in self.letter_buttons:
                self.letter_buttons[letter].button_style = 'success'
            
            # Update guessed word
            new_guessed = list(self.guessed_word)
            indices = [i for i, c in enumerate(self.current_word) if c == letter]
            for i in indices:
                new_guessed[i] = letter
            self.guessed_word = ''.join(new_guessed)
            
            self.message_display.value = f"<p style='color:green;'>✓ Correct! '{letter.upper()}' found at position(s): {[i+1 for i in indices]}</p>"
            
            # Check if word is complete
            if '_' not in self.guessed_word:
                self.message_display.value = f"<p style='color:green; font-weight:bold;'>🎉 You guessed '{self.current_word}' in {self.tries} tries!</p>"
                self.current_word_idx += 1
                if self.current_word_idx < len(self.words):
                    self.message_display.value += "<p>Click any button to start next word...</p>"
                    # Disable all buttons temporarily
                    for btn in self.letter_buttons.values():
                        btn.disabled = True
                    # Auto-start next word after delay
                    import time
                    time.sleep(2)
                    self.start_new_word()
                else:
                    self.end_game()
        else:
            # Visual feedback on button
            if letter in self.letter_buttons:
                self.letter_buttons[letter].button_style = 'danger'
            
            self.message_display.value = f"<p style='color:red;'>✗ '{letter.upper()}' is not in the word. Try again!</p>"
            
        self.letter_input.value = ''
        self.update_display()
        
    def update_display(self):
        # Word display with spacing
        word_html = ' '.join(f"<span style='font-size:28px; font-weight:bold; margin:5px; font-family:monospace;'>{c}</span>" 
                             for c in self.guessed_word)
        self.word_display.value = f"<div style='text-align:center; padding:20px; background:#f0f0f0; border-radius:10px;'>{word_html}</div>"
        
        # Stats
        self.stats_display.value = f"<p style='text-align:center;'><b>Tries:</b> {self.tries} | <b>Letters used:</b> {26 - len(self.letters_left)}/26</p>"
        
    def end_game(self):
        self.game_info.value = "<h3 style='color:green;'>🎊 Game Complete! 🎊</h3>"
        self.message_display.value = f"<p style='font-size:18px; text-align:center;'><b>You guessed all {len(self.words)} words!</b></p>"
        self.letter_input.disabled = True
        self.guess_btn.disabled = True
        for btn in self.letter_buttons.values():
            btn.disabled = True
        self.word_display.value = "<p style='text-align:center; font-size:24px; color:green;'>🏆 Well done! 🏆</p>"
        
    def display(self):
        display(self.main_ui)

# Create and display the game
game = WordGuessingGame()
game.display()